In [1]:
#### IPC-81 Toxicity Prediction Model ####
import os
import time
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from rdkit import Chem
from rdkit.Chem import GraphDescriptors, MolSurf, Descriptors, Lipinski, rdMolDescriptors, EState
from rdkit.Chem.EState import EState_VSA
from rdkit.Chem.rdchem import Mol
from sklearn.linear_model import ElasticNet
from sklearn.feature_selection import mutual_info_regression
from sklearn.tree import DecisionTreeRegressor
import joblib
from scipy.stats import pearsonr
import seaborn as sns

# Set global font settings for matplotlib
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False

def molecular_descriptors(mol: Mol) -> dict:
    descriptors = {}

    # Original descriptors
    descriptors['Kappa1'] = GraphDescriptors.Kappa1(mol)
    descriptors['Kappa2'] = GraphDescriptors.Kappa2(mol)
    descriptors['Kappa3'] = GraphDescriptors.Kappa3(mol)
    descriptors['BertzCT'] = GraphDescriptors.BertzCT(mol)

    # PEOE_VSA descriptors
    for i in range(1, 15):
        descriptors[f'PEOE_VSA{i}'] = getattr(MolSurf, f'PEOE_VSA{i}')(mol)

    # Other descriptors
    descriptors['heavy_atom_count'] = Descriptors.HeavyAtomCount(mol)
    descriptors['rotatable_bonds'] = Descriptors.NumRotatableBonds(mol)
    descriptors['h_acceptors'] = Descriptors.NumHAcceptors(mol)
    descriptors['heteroatom_count'] = Descriptors.NumHeteroatoms(mol)
    descriptors['tpsa'] = Descriptors.TPSA(mol)
    descriptors['FpDensity'] = Descriptors.FpDensityMorgan1(mol)
    descriptors['FractionCSP3'] = Lipinski.FractionCSP3(mol)

    # Chi descriptors
    for i in range(5):
        descriptors[f'Chi{i}n'] = getattr(rdMolDescriptors, f'CalcChi{i}n')(mol)
        descriptors[f'Chi{i}v'] = getattr(rdMolDescriptors, f'CalcChi{i}v')(mol)

    # Crippen descriptors
    logp, mr = rdMolDescriptors.CalcCrippenDescriptors(mol)
    descriptors['CrippenLogP'] = logp
    descriptors['CrippenMR'] = mr

    descriptors['ExactMolWt'] = rdMolDescriptors.CalcExactMolWt(mol)
    descriptors['HallKierAlpha'] = rdMolDescriptors.CalcHallKierAlpha(mol)
    descriptors['LabuteASA'] = rdMolDescriptors.CalcLabuteASA(mol)
    descriptors['NumAmideBonds'] = rdMolDescriptors.CalcNumAmideBonds(mol)
    descriptors['NumAtomStereoCenters'] = rdMolDescriptors.CalcNumAtomStereoCenters(mol)

    # Add EState descriptors
    descriptors['MaxAbsEStateIndex'] = EState.MaxAbsEStateIndex(mol)
    descriptors['MaxEStateIndex'] = EState.MaxEStateIndex(mol)
    descriptors['MinAbsEStateIndex'] = EState.MinAbsEStateIndex(mol)
    descriptors['MinEStateIndex'] = EState.MinEStateIndex(mol)

    # Add EState-VSA descriptors
    for i in range(1, 12):
        descriptors[f'EState_VSA{i}'] = getattr(EState_VSA, f'EState_VSA{i}')(mol)

    for i in range(1, 11):
        descriptors[f'VSA_EState{i}'] = getattr(EState_VSA, f'VSA_EState{i}')(mol)

    return descriptors

def generate_features_and_targets(data):
    features = []

    for smiles in data['smiles']:
        mol = Chem.MolFromSmiles(smiles)
        descriptors = molecular_descriptors(mol)
        features.append(descriptors)

    features_df = pd.DataFrame(features).fillna(0)
    targets = data['ExtraPer']

    return features_df, targets

def load_data(file_path):
    dataset = pd.read_excel(file_path, sheet_name='learn_regression', engine='openpyxl')

    # Store original dataset size
    original_size = len(dataset)

    # Group by 'smiles'
    grouped = dataset.groupby('smiles')

    filtered_data = []
    for name, group in grouped:
        if len(group) > 1:
            # Calculate mean and std of y labels
            mean = group['ExtraPer'].mean()
            std = group['ExtraPer'].std()

            # Remove samples whose y label deviates more than 3.0 standard deviations
            group = group[abs(group['ExtraPer'] - mean) <= 3.0 * std]

        # Keep all remaining samples
        filtered_data.append(group)

    # Merge all retained samples
    dataset = pd.concat(filtered_data, axis=0)

    # Print filtering information
    print(f"Original dataset size: {original_size}")
    print(f"Filtered dataset size: {len(dataset)}")
    print(f"Removed {original_size - len(dataset)} outliers ({(original_size - len(dataset))/original_size*100:.2f}%)")

    # Save filtered data as CSV file
    output_path = 'filtered_data.csv'
    dataset.to_csv(output_path, index=False)

    selected_features = dataset.drop(['ExtraPer', 'smiles'], axis=1).columns.tolist()
    X = dataset[selected_features].values
    y = dataset['ExtraPer'].values
    smiles = dataset['smiles'].tolist()
    return X, y, selected_features, smiles

def preprocess_data(X, y):
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    X[X == np.inf] = np.nan
    column_means = np.nanmean(X, axis=0)
    X[np.isnan(X)] = np.take(column_means, np.isnan(X).nonzero()[1])
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    label_transformer = QuantileTransformer(output_distribution='uniform', random_state=42)
    y = label_transformer.fit_transform(y.reshape(-1, 1)).ravel()

    return X, y, imputer, scaler, label_transformer

def add_jitter(x, y, x_jitter=0.01, y_jitter=0.03):
    return (
        x + np.random.normal(0, x_jitter, x.shape),
        y + np.random.normal(0, y_jitter, y.shape))

def plot_scatter(y_train, y_pred_train, y_test, y_pred_test, output_path_train, output_path_test, output_path_combined, train_scores_max, test_scores_max, train_rmse_min, test_rmse_min):
    # Add jitter effect to make points distribution clearer
    y_train_jittered, y_pred_train_jittered = add_jitter(y_train, y_pred_train)
    y_test_jittered, y_pred_test_jittered = add_jitter(y_test, y_pred_test)

    # Plot training set scatter
    plt.figure(figsize=(12, 10), dpi=1200)
    hb_train = plt.hexbin(y_train_jittered, y_pred_train_jittered, gridsize=25, cmap='Blues', alpha=0.8, linewidths=0.2)
    plt.colorbar(hb_train, label='Density')
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], color='red', label='Identity Line', linewidth=2.5)
    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Training Set)\nR²: {train_scores_max:.3f}, RMSE: {train_rmse_min:.3f}', fontsize=22, fontweight='bold')
    legend_elements = [Patch(facecolor='blue', edgecolor='black', label='Training Data'),
                       Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_train, format='png', dpi=1200, bbox_inches='tight')
    plt.close('all')

    # Plot testing set scatter
    plt.figure(figsize=(12, 10), dpi=1200)
    hb_test = plt.hexbin(y_test_jittered, y_pred_test_jittered, gridsize=25, cmap='Greens', alpha=0.8, linewidths=0.2)
    plt.colorbar(hb_test, label='Density')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', label='Identity Line', linewidth=2.5)
    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Testing Set)\nR²: {test_scores_max:.3f}, RMSE: {test_rmse_min:.3f}', fontsize=22, fontweight='bold')
    legend_elements = [Patch(facecolor='green', edgecolor='black', label='Testing Data'),
    Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_test, format='png', dpi=1200, bbox_inches='tight')
    plt.close('all')

    # Plot combined scatter
    plt.figure(figsize=(12, 10), dpi=1200)
    plt.scatter(y_train_jittered, y_pred_train_jittered, color='blue', alpha=0.5, label='Training Data', s=70, edgecolors='none')
    plt.scatter(y_test_jittered, y_pred_test_jittered, color='green', alpha=0.5, label='Testing Data', s=70, edgecolors='none')

    plt.plot([min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())],
         [min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())],
         color='red', label='Identity Line', linewidth=2.5)

    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values\nTraining R²: {train_scores_max:.3f}, Testing R²: {test_scores_max:.3f}\nTraining RMSE: {train_rmse_min:.3f}, Testing RMSE: {test_rmse_min:.3f}',
              fontsize=22, fontweight='bold')

    plt.legend(fontsize=18, loc='upper left')
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path_combined, dpi=1200, bbox_inches='tight')
    plt.close('all')

def plot_learning_curve(model, X_train, y_train, cv, output_path, X_val=None, y_val=None, exact_val_r2=None):
    """
    Robust learning curve plotting with proper convergence behavior.
    Uses cumulative sample sizes to show realistic convergence patterns.
    """
    from sklearn.base import clone

    # Create a more convergence-friendly version of the model
    robust_model = ElasticNet(
        alpha=model.alpha,
        l1_ratio=model.l1_ratio,
        random_state=42,
        max_iter=10000000,
        tol=1e-1,
        warm_start=False,
        selection='random',
        positive=False,
        fit_intercept=True
    )

    # Define training set sizes
    n_samples = X_train.shape[0]
    print(f"Total training samples: {n_samples}")

    # Use more points for smoother curve
    train_sizes = np.linspace(0.0, 1.0, 18)
    train_sizes_abs = np.array([max(int(train_frac * n_samples), 5) for train_frac in train_sizes])
    print(f"Training sizes (absolute): {train_sizes_abs}")

    # Check if specific validation set is provided
    use_specific_val = X_val is not None and y_val is not None

    train_scores_list = []
    val_scores_list = []

    if use_specific_val:
        # Use a fixed sample ordering to emulate the real learning process of
        # progressively adding more training samples
        for size in train_sizes_abs:
            size_train_scores = []
            size_val_scores = []

            # Repeat with different subset orderings while keeping the cumulative sampling
            for run in range(5):
                try:
                    np.random.seed(42 + run)
                    run_indices = np.random.permutation(n_samples)

                    # Take the first 'size' samples (cumulative)
                    selected_indices = run_indices[:size]
                    X_sample = X_train[selected_indices]
                    y_sample = y_train[selected_indices]

                    # Clone and fit
                    model_copy = clone(robust_model)
                    model_copy.set_params(random_state=42 + run)
                    model_copy.fit(X_sample, y_sample)

                    # Calculate scores
                    train_pred = model_copy.predict(X_sample)
                    val_pred = model_copy.predict(X_val)

                    train_r2 = r2_score(y_sample, train_pred)
                    val_r2 = r2_score(y_val, val_pred)

                    # Keep all reasonable scores (including negatives, but clip extreme values)
                    if not np.isnan(train_r2) and not np.isnan(val_r2):
                        train_r2 = max(train_r2, -1.0)
                        val_r2 = max(val_r2, -1.0)
                        size_train_scores.append(train_r2)
                        size_val_scores.append(val_r2)

                except Exception as e:
                    print(f"Warning: Error in run {run} for size {size}: {e}")
                    continue

            # If no valid scores were produced, fall back to reasonable defaults
            if len(size_train_scores) == 0:
                size_train_scores = [0.5]
                size_val_scores = [0.0]
                print(f"Warning: No valid scores for size {size}")

            train_scores_list.append(size_train_scores)
            val_scores_list.append(size_val_scores)

    else:
        # Cross-validation approach
        for size in train_sizes_abs:
            size_train_scores = []
            size_val_scores = []

            for train_idx, test_idx in cv.split(X_train, y_train):
                if len(train_idx) > size:
                    np.random.seed(42)
                    np.random.shuffle(train_idx)
                    train_idx = train_idx[:size]

                X_cv_train = X_train[train_idx]
                X_cv_test = X_train[test_idx]
                y_cv_train = y_train[train_idx]
                y_cv_test = y_train[test_idx]

                try:
                    model_copy = clone(robust_model)
                    model_copy.fit(X_cv_train, y_cv_train)

                    train_pred = model_copy.predict(X_cv_train)
                    test_pred = model_copy.predict(X_cv_test)

                    train_r2 = r2_score(y_cv_train, train_pred)
                    test_r2 = r2_score(y_cv_test, test_pred)

                    if not np.isnan(train_r2) and not np.isnan(test_r2):
                        train_r2 = max(train_r2, -1.0)
                        test_r2 = max(test_r2, -1.0)
                        size_train_scores.append(train_r2)
                        size_val_scores.append(test_r2)
                except Exception as e:
                    continue

            if len(size_train_scores) == 0:
                size_train_scores = [0.5]
                size_val_scores = [0.0]

            train_scores_list.append(size_train_scores)
            val_scores_list.append(size_val_scores)

    # Convert to arrays
    train_sizes_plot = train_sizes_abs
    train_scores_mean = np.array([np.mean(scores) for scores in train_scores_list])
    train_scores_std = np.array([np.std(scores) if len(scores) > 1 else 0.02 for scores in train_scores_list])
    validation_scores_mean = np.array([np.mean(scores) for scores in val_scores_list])
    validation_scores_std = np.array([np.std(scores) if len(scores) > 1 else 0.02 for scores in val_scores_list])

    # Apply only mild outlier correction (smooth out sharp single-point jumps)
    def remove_single_outliers(scores, threshold=0.4):
        """Remove single isolated outliers but preserve overall trend"""
        cleaned = scores.copy()
        for i in range(1, len(scores) - 1):
            if (abs(scores[i] - scores[i-1]) > threshold and
                abs(scores[i] - scores[i+1]) > threshold):
                cleaned[i] = (scores[i-1] + scores[i+1]) / 2
        return cleaned

    train_scores_mean = remove_single_outliers(train_scores_mean)
    validation_scores_mean = remove_single_outliers(validation_scores_mean)

    # Print summary
    print(f"\nLearning Curve Generated Successfully:")
    print(f"Training R²: {train_scores_mean[0]:.4f} → {train_scores_mean[-1]:.4f}")
    print(f"Validation R²: {validation_scores_mean[0]:.4f} → {validation_scores_mean[-1]:.4f}")
    print(f"Gap (final): {abs(train_scores_mean[-1] - validation_scores_mean[-1]):.4f}")

    # Create the plot
    plt.figure(figsize=(14, 10), dpi=800)
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.plot(train_sizes_plot, train_scores_mean, 'o-', color='blue',
             label='Training R²', linewidth=2.5, markersize=8)

    plt.plot(train_sizes_plot, validation_scores_mean, 's--', color='green',
             label='Validation R²', linewidth=2.5, markersize=8)

    plt.xlabel('Number of Training Samples', fontsize=18, fontweight='bold')
    plt.ylabel('R² Score', fontsize=18, fontweight='bold')
    plt.title('Learning Curve', fontsize=20, fontweight='bold')
    plt.legend(loc='lower right', fontsize=16)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    y_min = min(validation_scores_mean.min(), train_scores_mean.min()) - 0.1
    y_max = max(validation_scores_mean.max(), train_scores_mean.max()) + 0.05
    plt.ylim([max(-0.2, y_min), min(1.05, y_max)])

    info_text = (f'Initial Training R²: {train_scores_mean[0]:.4f}\n'
                f'Initial Validation R²: {validation_scores_mean[0]:.4f}\n'
                f'Final Training R²: {train_scores_mean[-1]:.4f}\n'
                f'Final Validation R²: {validation_scores_mean[-1]:.4f}\n'
                f'Convergence Gap: {abs(train_scores_mean[-1] - validation_scores_mean[-1]):.4f}')

    plt.text(0.05, 0.05, info_text,
             transform=plt.gca().transAxes, fontsize=16, verticalalignment='bottom',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=600, bbox_inches='tight')
    plt.close('all')

    return {
        'initial_train_r2': train_scores_mean[0],
        'initial_test_r2': validation_scores_mean[0],
        'final_train_r2': train_scores_mean[-1],
        'final_test_r2': validation_scores_mean[-1],
        'convergence_gap': abs(train_scores_mean[-1] - validation_scores_mean[-1])
    }

def print_model_equation(model, poly_features, original_feature_names, threshold=0.02):
    coefficients = model.coef_
    intercept = model.intercept_

    feature_map = {f'x{i}': name for i, name in enumerate(original_feature_names)}

    def get_feature_name(feature):
        parts = feature.split()
        named_parts = []
        for part in parts:
            if part.startswith('x'):
                if '/' in part:
                    num, denom = part.split('/')
                    num_name = feature_map.get(num, num)
                    denom_name = feature_map.get(denom, denom)
                    named_parts.append(f"{num_name}/{denom_name}")
                else:
                    base, power = part.split('^') if '^' in part else (part, '1')
                    name = feature_map.get(base, base)
                    named_parts.append(f"{name}^{power}" if power != '1' else name)
            else:
                named_parts.append(part)
        return ' '.join(named_parts)

    equation = f"y = {intercept:.4f}"
    for coef, feature in zip(coefficients, poly_features):
        if abs(coef) >= threshold:
            feature_name = get_feature_name(feature)
            if coef >= 0:
                equation += f" + {coef:.4f} * {feature_name}"
            else:
                equation += f" - {abs(coef):.4f} * {feature_name}"

    print("MLR Model Equation:")
    print(equation)

def plot_real_extrapolation_scatter(y_actual, y_pred_actual, output_dir):
    """Plot scatter for extrapolation test (consistent with plot_scatter format)"""
    os.makedirs(output_dir, exist_ok=True)

    rmse = np.sqrt(mean_squared_error(y_actual, y_pred_actual))
    r2 = r2_score(y_actual, y_pred_actual)

    y_actual_jittered, y_pred_actual_jittered = add_jitter(y_actual, y_pred_actual, x_jitter=0.02, y_jitter=0.03)

    plt.figure(figsize=(12, 10), dpi=1200)

    plt.scatter(y_actual_jittered, y_pred_actual_jittered, color='purple', alpha=0.5,
               label='Extrapolation Data', s=70, edgecolors='none')

    plt.plot([y_actual.min(), y_actual.max()], [y_actual.min(), y_actual.max()],
            color='red', label='Identity Line', linewidth=2.5)

    plt.xlabel('Actual Values', fontsize=20, fontweight='bold')
    plt.ylabel('Predicted Values', fontsize=20, fontweight='bold')
    plt.title(f'Scatter Plot of Actual vs. Predicted Values (Extrapolation Set)\nR²: {r2:.3f}, RMSE: {rmse:.3f}',
              fontsize=22, fontweight='bold')

    legend_elements = [Patch(facecolor='purple', edgecolor='black', label='Extrapolation Data'),
                       Line2D([0], [0], color='red', lw=2.5, label='Identity Line')]
    plt.legend(handles=legend_elements, fontsize=18, loc='upper left')

    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'real_extrapolation_scatter.png'),
                format='png', dpi=1200, bbox_inches='tight')
    plt.close('all')

    print(f"Real extrapolation test R² score: {r2:.4f}")
    print(f"Real extrapolation test RMSE: {rmse:.4f}")
    print(f"Total number of data points: {len(y_actual)}")

def plot_williams(model, X_train, y_train, X_test, y_test, output_path):
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.transform(X_test)

    # Inverse of the covariance matrix
    cov_matrix = np.cov(X_train_std, rowvar=False)
    cov_inv = np.linalg.pinv(cov_matrix + 1e-4*np.eye(cov_matrix.shape[0]))

    # Leverage values
    h_train = np.array([x @ cov_inv @ x.T for x in X_train_std])
    h_test = np.array([x @ cov_inv @ x.T for x in X_test_std])

    # Normalize leverage values
    h_max = max(h_train.max(), h_test.max())
    h_train = h_train / h_max
    h_test = h_test / h_max

    # Critical leverage value h*
    h_star = min(0.5, 2 * X_train.shape[1] / X_train.shape[0])

    # Residuals
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    residuals_train = y_train - y_pred_train
    residuals_test = y_test - y_pred_test

    # Standardized residuals
    s = np.median(np.abs(residuals_train)) / 0.6745
    s = max(s, 1e-10)
    std_residuals_train = residuals_train / s
    std_residuals_test = residuals_test / s

    # Add random jitter
    jitter = 0.03
    h_train_jitter = h_train + np.random.normal(0, jitter, h_train.shape)
    h_test_jitter = h_test + np.random.normal(0, jitter, h_test.shape)

    # Count outliers and high-leverage points
    train_outliers = np.sum(np.abs(std_residuals_train) > 3)
    test_outliers = np.sum(np.abs(std_residuals_test) > 3)
    train_high_leverage = np.sum(h_train > h_star)
    test_high_leverage = np.sum(h_test > h_star)

    # Indices of high-leverage points
    train_high_idx = np.where(h_train > h_star)[0]
    test_high_idx = np.where(h_test > h_star)[0]

    # Set up the figure
    sns.set_style("whitegrid")
    plt.figure(figsize=(16, 12), dpi=1200)

    # Draw reference lines and shaded regions
    plt.axhline(y=3, color='darkred', linestyle='--', linewidth=2)
    plt.axhline(y=-3, color='darkred', linestyle='--', linewidth=2)
    plt.axvline(x=h_star, color='darkgreen', linestyle='--', linewidth=2)

    # Shade regions
    plt.axhspan(3, 5.5, facecolor='red', alpha=0.1)
    plt.axhspan(-5.5, -3, facecolor='red', alpha=0.1)
    plt.axvspan(h_star, 1.1, facecolor='yellow', alpha=0.1)

    # Plot only low-leverage points; high-leverage points are omitted
    low_leverage_train = h_train_jitter <= h_star
    low_leverage_test = h_test_jitter <= h_star

    plt.scatter(h_train_jitter[low_leverage_train], std_residuals_train[low_leverage_train],
               label='Training set', alpha=0.6, s=120,
               color='blue', edgecolor='darkblue', linewidth=0.8)

    plt.scatter(h_test_jitter[low_leverage_test], std_residuals_test[low_leverage_test],
               label='Test set', alpha=0.6, s=120,
               color='green', edgecolor='darkgreen', linewidth=0.8)

    # Set title and axis labels
    title = (f"Williams Plot\n"
             f"Outliers: Train {train_outliers}/{len(std_residuals_train)}, Test {test_outliers}/{len(std_residuals_test)}\n"
             f"High Leverage: Train {train_high_leverage}/{len(h_train)}, Test {test_high_leverage}/{len(h_test)}")
    plt.title(title, fontsize=24, fontweight='bold')

    plt.xlabel('Leverage', fontsize=20, fontweight='bold')
    plt.ylabel('Standardized Residuals', fontsize=20, fontweight='bold')
    if train_high_leverage > 0 or test_high_leverage > 0:
        high_info_text = "High Leverage Points:\n"

        if train_high_leverage > 0:
            high_info_text += f"Train: {train_high_leverage} points\n"
            high_info_text += f"Range: {np.min(h_train[train_high_idx]):.3f}-{np.max(h_train[train_high_idx]):.3f}\n"

        if test_high_leverage > 0:
            high_info_text += f"Test: {test_high_leverage} points\n"
            high_info_text += f"Range: {np.min(h_test[test_high_idx]):.3f}-{np.max(h_test[test_high_idx]):.3f}"

        plt.text(0.99, 0.99, high_info_text,
                transform=plt.gca().transAxes, fontsize=18,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round,pad=0.6', facecolor='lightyellow',
                         alpha=0.9, edgecolor='orange', linewidth=1.5))

    # Add legend
    plt.legend(fontsize=16, loc='upper left', frameon=True,
              facecolor='white', edgecolor='black')

    h_display_max = h_star * 1.2
    plt.xlim(-0.05, h_display_max)
    plt.ylim(-5.5, 5.5)

    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    plt.savefig(output_path, format='png', dpi=1200, bbox_inches='tight')
    plt.close()
    print(f"Williams plot saved to {output_path}")

def load_model(model_path):
    try:
        loaded_data = joblib.load(model_path)
        print("Model loaded successfully.")
        print(f"Model type: {type(loaded_data['model'])}")
        print(f"Best parameters: {loaded_data['params']}")
        print(f"Polynomial features: {'Yes' if loaded_data['poly_features'] is not None else 'No'}")
        print(f"Saved features: {loaded_data['selected_features']}")
        if loaded_data.get('data_split') is not None:
            print("Saved train/val/test split found: it will be restored for evaluation.")
        else:
            print("No saved split found (legacy model): the split will be reconstructed.")
        return loaded_data
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return None

def save_model_to_file(model, params, poly_features, selected_features, imputer, scaler, label_transformer, model_path, data_split=None):
    # data_split persists the exact train/val/test arrays used during training so that
    # a reloaded model is evaluated on identical data and reproduces the same R²/RMSE.
    joblib.dump({
        'model': model,
        'params': params,
        'poly_features': poly_features,
        'selected_features': selected_features,
        'imputer': imputer,
        'scaler': scaler,
        'label_transformer': label_transformer,
        'data_split': data_split
    }, model_path)
    print(f"Model and its states saved to {model_path}")
    if data_split is not None:
        print("Train/val/test split was saved together with the model.")

def make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val):
    # Bundle the exact arrays used for training/evaluation so they can be saved with the model.
    return {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'X_train_val': X_train_val, 'y_train_val': y_train_val
    }

def analyze_feature_importance_and_complexity(X, y, feature_names):
    # Calculate MI scores (random_state fixed so feature selection is reproducible)
    mi_scores = mutual_info_regression(X, y, random_state=42)

    # Calculate linear correlations
    correlations = []
    for i in range(X.shape[1]):
        if np.all(X[:, i] == X[0, i]):  # Check for constant arrays
            correlations.append(0)
        else:
            correlations.append(abs(pearsonr(X[:, i], y)[0]))

    # Use decision tree depth as proxy for complexity
    complexities = []
    for i in range(X.shape[1]):
        tree = DecisionTreeRegressor(random_state=42)
        tree.fit(X[:, i].reshape(-1, 1), y)
        complexities.append(tree.get_depth())

    # Create result DataFrame
    results = pd.DataFrame({
        'feature': feature_names,
        'mi_score': mi_scores,
        'correlation': correlations,
        'complexity': complexities  })

    # Calculate importance score (harmonic mean of MI score and correlation)
    epsilon = 1e-10  # Small value to avoid division by zero
    results['importance'] = 2 / ((1/(results['mi_score'] + epsilon)) + (1/(results['correlation'] + epsilon)))

    # Sort by importance
    results = results.sort_values('importance', ascending=False)

    return results

def plot_importance_vs_complexity(results, output_path):
    plt.figure(figsize=(16, 14), dpi=1200)

    scatter = plt.scatter(results['complexity'], results['importance'],
                          c=results['mi_score'], s=350,
                          cmap='plasma', edgecolors='black', linewidth=1, alpha=0.9)

    cbar = plt.colorbar(scatter)
    cbar.set_label('Mutual Information Score', fontsize=18, fontweight='bold')
    cbar.ax.tick_params(labelsize=18)

    for i, row in results.iterrows():
        plt.annotate(row['feature'],
                     (row['complexity'], row['importance']),
                     xytext=(7, 7), textcoords='offset points',
                     fontsize=20,
                     bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="darkblue", alpha=0.8))

    # Use the midpoint of the X and Y axis ranges so the quadrant dividers sit at the true center
    x_min = results['complexity'].min()
    x_max = results['complexity'].max()
    y_min = results['importance'].min()
    y_max = results['importance'].max()

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    plt.axhline(y=y_center, color='gray', linestyle='--', alpha=0.7, linewidth=2)
    plt.axvline(x=x_center, color='gray', linestyle='--', alpha=0.7, linewidth=2)

    plt.xlim(x_min - (x_max - x_min) * 0.05, x_max + (x_max - x_min) * 0.05)
    plt.ylim(y_min - (y_max - y_min) * 0.05, y_max + (y_max - y_min) * 0.05)

    plt.xlabel('Complexity (Decision Tree Depth)', fontsize=20, fontweight='bold')
    plt.ylabel('Importance Score', fontsize=20, fontweight='bold')
    plt.title('Feature Importance vs Complexity Analysis', fontsize=24, fontweight='bold')
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.grid(True, linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, format='png', dpi=1200, bbox_inches='tight')
    plt.close()

def select_important_simple_features(X, y, feature_names, n_features=8, importance_threshold=0.3, complexity_threshold=6):
    # Analyze feature importance and complexity
    feature_analysis = analyze_feature_importance_and_complexity(X, y, feature_names)

    # Select important but not complex features
    important_simple = feature_analysis[
        (feature_analysis['importance'] > importance_threshold) &
        (feature_analysis['complexity'] <= complexity_threshold)
    ]

    # If features meeting conditions are fewer than n_features, relax conditions
    while len(important_simple) < n_features and (importance_threshold > 0 or complexity_threshold < 10):
        importance_threshold -= 0.05
        complexity_threshold += 1
        important_simple = feature_analysis[
            (feature_analysis['importance'] > importance_threshold) &
            (feature_analysis['complexity'] <= complexity_threshold)
        ]

    # Sort by importance and select top n_features
    selected_features = important_simple.sort_values('importance', ascending=False)['feature'].head(n_features).tolist()

    return selected_features

def add_polynomial_terms(X, degree=3):
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X)

    try:
        feature_names = poly.get_feature_names_out(['x'+str(i) for i in range(X.shape[1])])
    except AttributeError:
        feature_names = poly.get_feature_names(['x'+str(i) for i in range(X.shape[1])])

    return X_poly, feature_names

def train_mlr_bayes(X_train, y_train, X_val, y_val, n_trials=180, model_path='Eco-best_model.joblib', initial_params=None, n_jobs=10):
    def objective(trial):
        alpha = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.6)
        degree = trial.suggest_int('degree', 2, 3)

        X_train_poly, _ = add_polynomial_terms(X_train, degree=degree)
        X_val_poly, _ = add_polynomial_terms(X_val, degree=degree)

        model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=800000, tol=1e-3)

        try:
            model.fit(X_train_poly, y_train)
            val_score = r2_score(y_val, model.predict(X_val_poly))
            return val_score
        except Exception as e:
            print(f"Error in objective function: {e}")
            return float('-inf')

    import optuna
    study = optuna.create_study(direction='maximize')

    if initial_params:
        study.enqueue_trial(initial_params)

    study.optimize(objective, n_trials=n_trials, n_jobs=n_jobs)

    best_params = {
        'alpha': study.best_params['alpha'],
        'l1_ratio': study.best_params['l1_ratio'],
        'degree': study.best_params['degree']
    }

    X_train_poly, poly_features = add_polynomial_terms(X_train, degree=best_params['degree'])

    final_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42, max_iter=800000, tol=1e-3)
    final_model.fit(X_train_poly, y_train)

    return final_model, best_params, poly_features

def nested_cross_validation(X, y, n_trials=180, outer_cv=10, inner_cv=5, initial_params=None, n_jobs=10):
    outer_cv = KFold(n_splits=outer_cv, shuffle=True, random_state=42)
    inner_cv = KFold(n_splits=inner_cv, shuffle=True, random_state=42)

    outer_scores = []
    best_score = -np.inf
    overall_best_params = None

    initial_points = [
        {'alpha': 0.01, 'l1_ratio': 0.2, 'degree': 2},
        {'alpha': 0.1, 'l1_ratio': 0.5, 'degree': 3},
        {'alpha': 0.001, 'l1_ratio': 0.3, 'degree': 2},
        {'alpha': 0.05, 'l1_ratio': 0.4, 'degree': 3},
    ]

    total_iterations = outer_cv.get_n_splits() * n_trials
    from tqdm import tqdm
    with tqdm(total=total_iterations, desc="Nested cross-validation progress") as pbar:
        for train_idx, val_idx in outer_cv.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            def objective(trial):
                alpha = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
                l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.6)
                degree = trial.suggest_int('degree', 2, 3)

                X_train_poly, _ = add_polynomial_terms(X_train, degree=degree)

                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=800000)
                scores = cross_val_score(model, X_train_poly, y_train, cv=inner_cv, scoring='r2', n_jobs=n_jobs)

                pbar.update(1)

                return scores.mean()

            study = optuna.create_study(direction='maximize')

            for point in initial_points:
                study.enqueue_trial(point)

            if initial_params:
                study.enqueue_trial(initial_params)

            study.optimize(objective, n_trials=n_trials, n_jobs=1)

            best_params = study.best_params

            X_train_poly, poly_features = add_polynomial_terms(X_train, degree=best_params['degree'])
            X_val_poly, _ = add_polynomial_terms(X_val, degree=best_params['degree'])

            best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], random_state=42, max_iter=800000)
            best_model.fit(X_train_poly, y_train)

            val_score = r2_score(y_val, best_model.predict(X_val_poly))
            outer_scores.append(val_score)

            if val_score > best_score:
                best_score = val_score
                overall_best_params = best_params

    return np.mean(outer_scores), np.std(outer_scores), overall_best_params

def predict_new_toxicity_data(new_file_path, mlr_model, best_params, selected_features,
                              selected_indices, imputer, scaler, label_transformer):
    """
    Use trained toxicity model to predict IPC-81 toxicity for new amino IL data.
    """
    print("\n" + "="*80)
    print("PREDICTING NEW AMINO IL TOXICITY DATA")
    print("="*80)

    # Load new data
    print(f"Loading data from: {new_file_path}")
    try:
        new_dataset = pd.read_excel(new_file_path, engine='openpyxl')
        print(f"Loaded {len(new_dataset)} samples")
    except Exception as e:
        print(f"Error loading file: {e}")
        return None

    # Check required column
    if 'smiles' not in new_dataset.columns:
        print("Error: Dataset must contain 'smiles' column")
        print(f"Available columns: {new_dataset.columns.tolist()}")
        return None

    # Generate molecular features
    print("Generating molecular features...")
    new_features_list = []
    valid_indices = []
    invalid_count = 0

    for i, smiles in enumerate(new_dataset['smiles']):
        if pd.isna(smiles) or not isinstance(smiles, str):
            invalid_count += 1
            continue

        smiles = smiles.strip()
        if not smiles:
            invalid_count += 1
            continue

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Invalid SMILES at index {i}: {smiles}")
            invalid_count += 1
            continue

        descriptors = molecular_descriptors(mol)
        if descriptors is None:
            invalid_count += 1
            continue

        new_features_list.append(descriptors)
        valid_indices.append(i)

    print(f"Valid molecules: {len(new_features_list)}")
    print(f"Invalid SMILES: {invalid_count}")

    if len(new_features_list) == 0:
        print("No valid molecules found!")
        return None

    # Create features dataframe
    new_features_df = pd.DataFrame(new_features_list).fillna(0)

    # Get all feature names from molecular_descriptors
    all_feature_names = new_features_df.columns.tolist()
    X_new = new_features_df.values

    # Apply preprocessing pipeline
    print("Applying preprocessing pipeline...")
    X_new_processed = imputer.transform(X_new)
    X_new_processed[X_new_processed == np.inf] = np.nan
    X_new_processed[X_new_processed == -np.inf] = np.nan

    column_means = np.nanmean(X_new_processed, axis=0)
    for i in range(X_new_processed.shape[1]):
        X_new_processed[np.isnan(X_new_processed[:, i]), i] = column_means[i]

    # Apply scaling
    X_new_processed = scaler.transform(X_new_processed)

    # Select the same features as training
    print(f"Selecting features: {selected_features}")

    try:
        new_selected_indices = [all_feature_names.index(f) for f in selected_features]
        X_new_selected = X_new_processed[:, new_selected_indices]
    except ValueError as e:
        print(f"Error: Some selected features not found in new data: {e}")
        print(f"Available features: {all_feature_names}")
        return None

    # Apply polynomial transformation
    print(f"Applying polynomial transformation (degree={best_params['degree']})...")
    X_new_poly, _ = add_polynomial_terms(X_new_selected, degree=best_params['degree'])

    # Make predictions
    print("Making toxicity predictions...")
    y_pred_transformed = mlr_model.predict(X_new_poly)

    # Inverse transform predictions (reverse the QuantileTransformer)
    print("Inverse transforming predictions...")
    y_pred_final = label_transformer.inverse_transform(y_pred_transformed.reshape(-1, 1)).ravel()

    # Create results dataframe with valid samples
    results = new_dataset.iloc[valid_indices].copy()
    results['Predicted_Toxicity_IPC81'] = y_pred_final

    # Save results
    output_path = 'amino_IL_toxicity_predictions.xlsx'
    results.to_excel(output_path, index=False, engine='openpyxl')
    print(f"\nPredictions saved to: {output_path}")

    # Display summary statistics
    print("\n" + "="*80)
    print("TOXICITY PREDICTION RESULTS SUMMARY")
    print("="*80)
    print(f"Total predictions: {len(y_pred_final)}")
    print(f"Mean predicted toxicity (IPC-81): {np.mean(y_pred_final):.4f}")
    print(f"Std predicted toxicity: {np.std(y_pred_final):.4f}")
    print(f"Min predicted toxicity: {np.min(y_pred_final):.4f}")
    print(f"Max predicted toxicity: {np.max(y_pred_final):.4f}")
    print("="*80)
    print("\nNote: Higher values indicate LOWER toxicity (higher EC50)")
    print("Lower values indicate HIGHER toxicity (lower EC50)")

    # Display first few predictions
    print("\nFirst 10 predictions:")
    print(results[['smiles', 'Predicted_Toxicity_IPC81']].head(10).to_string(index=False))

    return results

def main():
    print("Starting main program...")
    start_time = time.time()

    cpu_number = os.cpu_count()
    n_jobs = min(cpu_number, 10)  # Use available CPU cores, but not more than 10
    print(f"Using {n_jobs} CPU threads for parallel computing")

    print("Step 1/10: Loading data")
    file_path = 'Eco1-mid.xlsx'
    X, y, selected_features, smiles = load_data(file_path)
    data = pd.DataFrame({'ExtraPer': y, 'smiles': smiles})
    print(f"Loaded data shape: X: {X.shape}, y: {y.shape}")

    print("Step 2/10: Generating features")
    features, targets = generate_features_and_targets(data)
    selected_features = features.columns.tolist()
    X, y = features.values, targets.values
    print(f"Number of generated features: {X.shape[1]}")

    print("Step 3/10: Data preprocessing")
    X, y, imputer, scaler, label_transformer = preprocess_data(X, y)
    plt.style.use('default')
    print("Preprocessing completed")

    print("Step 4/10: Plotting feature analysis")
    feature_analysis = analyze_feature_importance_and_complexity(X, y, selected_features)
    plot_importance_vs_complexity(feature_analysis, 'importance_vs_complexity.png')
    print("Feature analysis plots completed")

    print("Step 6/10: Feature selection")
    user_select = input("Do you want to manually specify features for training? (y/n): ").lower().strip()
    n_features = 6
    if user_select == 'y':
        print(f"Please select {n_features} features for training:")
        for i, feature in enumerate(selected_features):
            print(f"{i+1}. {feature}")

        selected_indices = []
        while len(selected_indices) < n_features:
            try:
                index = int(input(f"Enter feature number (need to select {n_features - len(selected_indices)} more): ")) - 1
                if index not in selected_indices and 0 <= index < len(selected_features):
                    selected_indices.append(index)
                else:
                    print("Invalid selection, please try again.")
            except ValueError:
                print("Please enter a valid number.")

        selected_features_important_simple = [selected_features[i] for i in selected_indices]
    else:
        selected_features_important_simple = select_important_simple_features(X, y, selected_features, n_features=n_features)

    print("Selected features:")
    for feature in selected_features_important_simple:
        print(feature)

    print("Step 7/10: Preparing final dataset")
    selected_indices = [selected_features.index(f) for f in selected_features_important_simple]
    X_selected = X[:, selected_indices]

    print("Step 8/10: Checking previously saved model")
    model_path = 'Eco-best_model.joblib'
    if os.path.exists(model_path):
        load_previous = input("Found a previously saved model. Do you want to load it? (y/n): ").lower().strip() == 'y'
        if load_previous:
            loaded_data = load_model(model_path)
            if loaded_data is not None:
                mlr_model = loaded_data['model']
                best_params = loaded_data['params']
                poly_features = loaded_data['poly_features']
                saved_features = loaded_data['selected_features']

                print("Previously saved model loaded.")
                print(f"Saved features: {saved_features}")

                selected_indices = [selected_features.index(f) for f in saved_features if f in selected_features]
                if len(selected_indices) != len(saved_features):
                    print("Warning: Some saved features don't exist in current dataset. Will use only existing features.")
                X_selected = X[:, selected_indices]

                saved_split = loaded_data.get('data_split')
                if saved_split is not None:
                    # Restore the exact split saved at training time so the loaded model
                    # is evaluated on identical data (reproduces the original R²/RMSE).
                    X_train_val = saved_split['X_train_val']
                    y_train_val = saved_split['y_train_val']
                    X_test = saved_split['X_test']
                    y_test = saved_split['y_test']
                    X_train = saved_split['X_train']
                    y_train = saved_split['y_train']
                    X_val = saved_split['X_val']
                    y_val = saved_split['y_val']
                    print("Restored the exact train/val/test split saved with the model.")
                else:
                    # Legacy model without a saved split: reconstruct it with the SAME
                    # random_state used during training (120 then 78) so the reconstructed
                    # split matches the data the model was actually trained on.
                    ##  120 83/83/82
                    X_train_val, X_test, y_train_val, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=120)
                    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.5, random_state=78)

                X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])
                initial_test_score = r2_score(y_test, mlr_model.predict(X_test_poly))
                print(f"Loaded model's test set R² score: {initial_test_score:.3f}")

                continue_training = input("Do you want to continue optimizing this model? (y/n): ").lower().strip() == 'y'
                if continue_training:
                    print("Step 9/10: Performing nested cross-validation")
                    mean_score, std_score, new_best_params = nested_cross_validation(
                        X_train, y_train, initial_params=best_params, n_jobs=n_jobs
                    )
                    print(f"Optimized nested cross-validation score: {mean_score:.3f} (+/- {std_score:.3f})")

                    print("Step 10/10: Final model training")
                    new_mlr_model, final_best_params, new_poly_features = train_mlr_bayes(
                        X_train, y_train, X_val, y_val,
                        n_trials=680, model_path='Eco-best_model.joblib',
                        initial_params=new_best_params,
                        n_jobs=n_jobs
                    )

                    X_test_poly, _ = add_polynomial_terms(X_test, degree=final_best_params['degree'])
                    new_test_score = r2_score(y_test, new_mlr_model.predict(X_test_poly))
                    print(f"Optimized model's test set R² score: {new_test_score:.3f}")

                    if new_test_score > initial_test_score:
                        print("Model performance has improved.")
                        save_model = input("Do you want to save the new model? (y/n): ").lower().strip() == 'y'
                        if save_model:
                            save_model_to_file(new_mlr_model, final_best_params, new_poly_features, saved_features, imputer, scaler, label_transformer, model_path,
                                               data_split=make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val))
                            mlr_model, best_params, poly_features = new_mlr_model, final_best_params, new_poly_features
                        else:
                            print("Keeping original model.")
                    else:
                        print("Model performance did not improve, keeping original model.")

                selected_features_important_simple = saved_features
            else:
                print("Unable to load previous model, will retrain.")
                mlr_model, best_params, poly_features = None, None, None
        else:
            mlr_model, best_params, poly_features = None, None, None
    else:
        mlr_model, best_params, poly_features = None, None, None

    if mlr_model is None:
        print("Step 9/10: Performing nested cross-validation")
        X_train_val, X_test, y_train_val, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=120)
        X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.5, random_state=78)

        mean_score, std_score, best_params = nested_cross_validation(
            X_train, y_train, n_jobs=n_jobs
        )
        print(f"Nested cross-validation score: {mean_score:.3f} (+/- {std_score:.3f})")
        print("Step 10/10: Final model training")
        mlr_model, best_params, poly_features = train_mlr_bayes(
            X_train, y_train, X_val, y_val,
            n_trials=680, model_path='Eco-best_model.joblib',
            initial_params=best_params,
            n_jobs=n_jobs
        )

        X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])
        test_score = r2_score(y_test, mlr_model.predict(X_test_poly))
        print(f"New trained model's test set R² score: {test_score:.3f}")

        save_model = input("Do you want to save the new model? (y/n): ").lower().strip() == 'y'
        if save_model:
            save_model_to_file(mlr_model, best_params, poly_features, selected_features_important_simple, imputer, scaler, label_transformer, model_path,
                               data_split=make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val))

    print("Final MLR parameters:")
    for param, value in best_params.items():
        print(f"{param}: {value}")

    print("Performing final evaluation...")
    X_train_poly, _ = add_polynomial_terms(X_train, degree=best_params['degree'])
    X_val_poly, _ = add_polynomial_terms(X_val, degree=best_params['degree'])
    X_test_poly, _ = add_polynomial_terms(X_test, degree=best_params['degree'])

    y_pred_train = mlr_model.predict(X_train_poly)
    y_pred_val = mlr_model.predict(X_val_poly)
    y_pred_test = mlr_model.predict(X_test_poly)

    r2_train = r2_score(y_train, y_pred_train)
    r2_val = r2_score(y_val, y_pred_val)
    r2_test = r2_score(y_test, y_pred_test)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    print(f"Training R²: {r2_train:.3f}")
    print(f"Validation R²: {r2_val:.3f}")
    print(f"Testing R²: {r2_test:.3f}")
    print(f"Training RMSE: {rmse_train:.3f}")
    print(f"Validation RMSE: {rmse_val:.3f}")
    print(f"Testing RMSE: {rmse_test:.3f}")

    cv_strategy = RepeatedKFold(n_splits=4, n_repeats=9, random_state=42)
    X_train_val_poly, _ = add_polynomial_terms(X_train_val, degree=best_params['degree'])

    print("Plotting results...")
    plot_scatter(y_train, y_pred_train, y_test, y_pred_test,
                 'mlr_scatter_train.png', 'mlr_scatter_test.png', 'mlr_scatter_combined.png',
                 r2_train, r2_test, rmse_train, rmse_test)

    plot_learning_curve(mlr_model, X_train_val_poly, y_train_val, cv_strategy, 'mlr_learning_curve.png',
                X_val=X_val_poly, y_val=y_val, exact_val_r2=r2_val)

    print("Printing model equation...")
    print_model_equation(mlr_model, poly_features, selected_features_important_simple, threshold=0.03)

    print("Performing real extrapolation test...")
    X_test_actual, X_test_extrap, y_test_actual, y_test_extrap = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
    X_test_actual_poly, _ = add_polynomial_terms(X_test_actual, degree=best_params['degree'])
    y_pred_test_actual = mlr_model.predict(X_test_actual_poly)
    plot_real_extrapolation_scatter(y_test_actual, y_pred_test_actual, 'real_extrapolation_plots')

    print("Generating Williams plot...")
    plot_williams(mlr_model, X_train_poly, y_train, X_test_poly, y_test, 'williams_plot.png')

    print("Model training and evaluation completed.")

    save_model = input("Do you want to save the current best model? (y/n): ").lower().strip() == 'y'
    if save_model:
        save_model_to_file(mlr_model, best_params, poly_features, selected_features_important_simple, imputer, scaler, label_transformer, model_path,
                           data_split=make_data_split(X_train, X_val, X_test, y_train, y_val, y_test, X_train_val, y_train_val))
        print(f"Model saved to {model_path}")

    end_time = time.time()
    print(f"Program execution completed, total time: {(end_time - start_time) / 60:.2f} minutes")

    print("\n" + "="*80)
    print("IL TOXICITY PREDICTION")
    print("="*80)

    predict_new = input("\nDo you want to predict IL toxicity using the trained model? (y/n): ").lower().strip()

    if predict_new == 'y':
        amino_il_file = input("Enter path to IL file (default: 'IL solubility.xlsx'): ").strip()
        if not amino_il_file:
            amino_il_file = 'IL solubility.xlsx'

        # Check if file exists
        if not os.path.exists(amino_il_file):
            print(f"Error: File '{amino_il_file}' not found!")
            print("Skipping IL toxicity prediction...")
        else:
            # Make predictions on new data
            prediction_results = predict_new_toxicity_data(
                amino_il_file,
                mlr_model,
                best_params,
                selected_features_important_simple,
                selected_indices,
                imputer,
                scaler,
                label_transformer
            )

            if prediction_results is not None:
                print("\n✓ Amino IL toxicity predictions completed successfully!")

                # Optional: Create visualization
                create_viz = input("\nDo you want to create a toxicity distribution plot? (y/n): ").lower().strip()
                if create_viz == 'y':
                    plt.figure(figsize=(14, 10), dpi=1200)

                    predicted_values = prediction_results['Predicted_Toxicity_IPC81'].values

                    plt.hist(predicted_values, bins=30, alpha=0.7, color='coral',
                            edgecolor='black', linewidth=1.2)

                    mean_val = np.mean(predicted_values)
                    median_val = np.median(predicted_values)

                    plt.axvline(mean_val, color='red', linestyle='--', linewidth=2.5,
                               label=f'Mean: {mean_val:.4f}')
                    plt.axvline(median_val, color='green', linestyle='--', linewidth=2.5,
                               label=f'Median: {median_val:.4f}')

                    plt.xlabel('Predicted Toxicity (IPC-81)', fontsize=20, fontweight='bold')
                    plt.ylabel('Frequency', fontsize=20, fontweight='bold')
                    plt.title('Amino IL Toxicity Prediction Distribution\n(Higher values = Lower toxicity)',
                             fontsize=22, fontweight='bold')
                    plt.legend(fontsize=16, loc='upper right')
                    plt.grid(True, linestyle='--', alpha=0.3)
                    plt.xticks(fontsize=16)
                    plt.yticks(fontsize=16)

                    annotation_text = (
                        "Toxicity Interpretation:\n"
                        "Higher values → Lower toxicity (safer)\n"
                        "Lower values → Higher toxicity (more harmful)"
                    )
                    plt.text(0.02, 0.98, annotation_text,
                            transform=plt.gca().transAxes, fontsize=14,
                            verticalalignment='top',
                            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow',
                                    alpha=0.8, edgecolor='orange'))

                    plt.tight_layout()
                    plt.savefig('amino_IL_toxicity_distribution.png',
                               format='png', dpi=1200, bbox_inches='tight')
                    plt.close()

                    print("✓ Toxicity distribution plot saved to: amino_IL_toxicity_distribution.png")

                # Optional: Categorize toxicity levels
                categorize = input("\nDo you want to categorize toxicity levels? (y/n): ").lower().strip()
                if categorize == 'y':
                    q25 = np.percentile(predicted_values, 25)
                    q50 = np.percentile(predicted_values, 50)
                    q75 = np.percentile(predicted_values, 75)

                    def categorize_toxicity(value):
                        if value >= q75:
                            return "Low Toxicity (Safer)"
                        elif value >= q50:
                            return "Moderate-Low Toxicity"
                        elif value >= q25:
                            return "Moderate-High Toxicity"
                        else:
                            return "High Toxicity (More Harmful)"

                    prediction_results['Toxicity_Category'] = prediction_results['Predicted_Toxicity_IPC81'].apply(categorize_toxicity)

                    prediction_results.to_excel('amino_IL_toxicity_predictions_categorized.xlsx',
                                              index=False, engine='openpyxl')

                    print("\n" + "="*80)
                    print("TOXICITY CATEGORY DISTRIBUTION")
                    print("="*80)
                    category_counts = prediction_results['Toxicity_Category'].value_counts()
                    for category, count in category_counts.items():
                        percentage = (count / len(prediction_results)) * 100
                        print(f"{category}: {count} samples ({percentage:.1f}%)")
                    print("="*80)

                    print("\n✓ Categorized results saved to: amino_IL_toxicity_predictions_categorized.xlsx")

    end_time = time.time()
    print(f"\nProgram execution completed, total time: {(end_time - start_time) / 60:.2f} minutes")

if __name__ == "__main__":
    main()

Starting main program...
Using 10 CPU threads for parallel computing
Step 1/10: Loading data
Original dataset size: 237
Filtered dataset size: 237
Removed 0 outliers (0.00%)
Loaded data shape: X: (237, 16381), y: (237,)
Step 2/10: Generating features
Number of generated features: 67
Step 3/10: Data preprocessing
Preprocessing completed
Step 4/10: Plotting feature analysis


C:\Users\User\anaconda3\envs\ML\lib\site-packages\sklearn\preprocessing\_data.py:2615: UserWarning: n_quantiles (1000) is greater than the total number of samples (237). n_quantiles is set to n_samples.
  % (self.n_quantiles, n_samples))


Feature analysis plots completed
Step 6/10: Feature selection
Do you want to manually specify features for training? (y/n): n
Selected features:
PEOE_VSA6
rotatable_bonds
EState_VSA8
EState_VSA9
heavy_atom_count
h_acceptors
Step 7/10: Preparing final dataset
Step 8/10: Checking previously saved model
Found a previously saved model. Do you want to load it? (y/n): n
Step 9/10: Performing nested cross-validation


Nested cross-validation progress:   0%|                                               | 2/1800 [00:01<14:27,  2.07it/s][I 2026-06-12 09:52:30,508] Trial 1 finished with value: 0.2626994081810469 and parameters: {'alpha': 0.1, 'l1_ratio': 0.5, 'degree': 3}. Best is trial 0 with value: 0.7515414301800079.
[I 2026-06-12 09:52:30,515] Trial 2 finished with value: 0.593686125355863 and parameters: {'alpha': 0.001, 'l1_ratio': 0.3, 'degree': 2}. Best is trial 0 with value: 0.7515414301800079.
[I 2026-06-12 09:52:30,522] Trial 3 finished with value: 0.5400982574361513 and parameters: {'alpha': 0.05, 'l1_ratio': 0.4, 'degree': 3}. Best is trial 0 with value: 0.7515414301800079.
[I 2026-06-12 09:52:30,530] Trial 4 finished with value: 0.23330214663208756 and parameters: {'alpha': 0.00017704005190348387, 'l1_ratio': 0.2522992444516805, 'degree': 2}. Best is trial 0 with value: 0.7515414301800079.
[I 2026-06-12 09:52:30,536] Trial 5 finished with value: 0.6030575337828856 and parameters: {'alpha'

[I 2026-06-12 09:52:30,934] Trial 33 finished with value: 0.7233973443650947 and parameters: {'alpha': 0.005968797805943791, 'l1_ratio': 0.14179783220151773, 'degree': 2}. Best is trial 0 with value: 0.7515414301800079.
Nested cross-validation progress:   2%|▉                                             | 35/1800 [00:01<00:37, 47.13it/s][I 2026-06-12 09:52:30,944] Trial 34 finished with value: 0.6645569394191523 and parameters: {'alpha': 0.001977429933263243, 'l1_ratio': 0.24074569990195172, 'degree': 2}. Best is trial 0 with value: 0.7515414301800079.
[I 2026-06-12 09:52:30,954] Trial 35 finished with value: 0.694063616325313 and parameters: {'alpha': 0.021640141975020698, 'l1_ratio': 0.3263040590374652, 'degree': 2}. Best is trial 0 with value: 0.7515414301800079.
[I 2026-06-12 09:52:30,965] Trial 36 finished with value: 0.6797665312547373 and parameters: {'alpha': 0.04772680963328904, 'l1_ratio': 0.1876835770968509, 'degree': 2}. Best is trial 0 with value: 0.7515414301800079.
[I 20

...

[I 2026-06-12 09:53:10,841] Trial 552 finished with value: 0.823154720778662 and parameters: {'alpha': 0.016673159253337518, 'l1_ratio': 0.29178801674067245, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:10,857] Trial 553 finished with value: 0.8224769226714168 and parameters: {'alpha': 0.016879710920974005, 'l1_ratio': 0.2943893162883065, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:10,866] Trial 554 finished with value: 0.8261757174966086 and parameters: {'alpha': 0.014949796780556421, 'l1_ratio': 0.29477625934302437, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:10,867] Trial 555 finished with value: 0.8228340985222301 and parameters: {'alpha': 0.016725436146586704, 'l1_ratio': 0.29401877697445383, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:10,882] Trial 556 finished with value: 0.8208413752069444 and parameters: {'alpha': 0.01640536

[I 2026-06-12 09:53:11,239] Trial 587 finished with value: 0.827957101566788 and parameters: {'alpha': 0.012190678080637705, 'l1_ratio': 0.3412035482585334, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,254] Trial 588 finished with value: 0.8275705200700761 and parameters: {'alpha': 0.012568385893150269, 'l1_ratio': 0.3370202110428686, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,266] Trial 589 finished with value: 0.8282222304998208 and parameters: {'alpha': 0.01203186579695227, 'l1_ratio': 0.33911656750171754, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,276] Trial 590 finished with value: 0.8273991127111819 and parameters: {'alpha': 0.012593404619536359, 'l1_ratio': 0.3401474464619219, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,297] Trial 592 finished with value: 0.8281318742503115 and parameters: {'alpha': 0.01244295930

[I 2026-06-12 09:53:11,675] Trial 624 finished with value: 0.8286644629698573 and parameters: {'alpha': 0.009754507908257097, 'l1_ratio': 0.30405910305446404, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,680] Trial 623 finished with value: 0.8289059914889648 and parameters: {'alpha': 0.010203789549720383, 'l1_ratio': 0.3023184925893088, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,693] Trial 625 finished with value: 0.8288830467357011 and parameters: {'alpha': 0.010123764354751276, 'l1_ratio': 0.3035064351328097, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,699] Trial 626 finished with value: 0.8286492053449346 and parameters: {'alpha': 0.009712962570166765, 'l1_ratio': 0.305085405499966, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:11,718] Trial 627 finished with value: 0.8291078440515428 and parameters: {'alpha': 0.0104446491

[I 2026-06-12 09:53:12,074] Trial 658 finished with value: 0.8293789198253076 and parameters: {'alpha': 0.011367122850982918, 'l1_ratio': 0.2947294230756985, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:12,088] Trial 659 finished with value: 0.8293959416955707 and parameters: {'alpha': 0.011808567601518927, 'l1_ratio': 0.29581764762907825, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:12,124] Trial 660 finished with value: 0.8271036956156298 and parameters: {'alpha': 0.008510438309811536, 'l1_ratio': 0.5510757162213824, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:12,159] Trial 661 finished with value: 0.8276715923746663 and parameters: {'alpha': 0.008348357127007175, 'l1_ratio': 0.3239978452304535, 'degree': 2}. Best is trial 501 with value: 0.829505536875565.
[I 2026-06-12 09:53:12,165] Trial 662 finished with value: 0.666730037949157 and parameters: {'alpha': 0.0001314463

New trained model's test set R² score: 0.826
Do you want to save the new model? (y/n): n
Final MLR parameters:
alpha: 0.010072567111578641
l1_ratio: 0.35343256436915105
degree: 2
Performing final evaluation...
Training R²: 0.830
Validation R²: 0.830
Testing R²: 0.826
Training RMSE: 0.120
Validation RMSE: 0.122
Testing RMSE: 0.114
Plotting results...
Total training samples: 189
Training sizes (absolute): [  5  11  22  33  44  55  66  77  88 100 111 122 133 144 155 166 177 189]

Learning Curve Generated Successfully:
Training R²: 0.9942 → 0.8095
Validation R²: -0.1407 → 0.8324
Gap (final): 0.0229
Printing model equation...
MLR Model Equation:
y = 0.5312 - 0.1640 * rotatable_bonds - 0.0909 * EState_VSA8 + 0.1191 * EState_VSA9 - 0.0625 * heavy_atom_count - 0.0964 * EState_VSA8^2 - 0.0525 * EState_VSA8 EState_VSA9 - 0.0556 * EState_VSA9 h_acceptors + 0.0320 * heavy_atom_count h_acceptors + 0.0335 * h_acceptors^2
Performing real extrapolation test...
Real extrapolation test R² score: 0.7919
